# Spam Classification — XGBoost

**Dataset:** SMS Spam Collection (ham/spam labeled).

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

All libraries imported successfully.


## 2. Load and Explore the Dataset

In [2]:
# Load the dataset
df = pd.read_csv('spam.csv', encoding='latin1')

# Keep only the relevant columns and rename them
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'message']

print('Dataset shape:', df.shape)
df.head()

Dataset shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# Class distribution
print('Class distribution:')
print(df['label'].value_counts())

# Check for missing values
print('\nMissing values:')
print(df.isnull().sum())

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


## 3. Data Preprocessing

Convert text to numerical features using **TF-IDF** vectorization. Encode labels as binary: `ham = 0`, `spam = 1`.

In [4]:
# Encode labels: ham=0, spam=1
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['message'])
y = df['label_encoded']

print('Feature matrix shape:', X.shape)
print('Label distribution:')
print(y.value_counts())

Feature matrix shape: (5572, 5000)
Label distribution:
label_encoded
0    4825
1     747
Name: count, dtype: int64


In [5]:
# Train/Test split (80% train, 20% test) — same split for all models
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

Training set size: 4457
Test set size: 1115


## 4. XGBoost Classifier

Gradient boosting with `scale_pos_weight` to handle class imbalance.

In [6]:
# Compute class imbalance weight
scale = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# Train XGBoost
xgb = XGBClassifier(
    scale_pos_weight=scale,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('=== XGBoost Classification Report ===')
print(classification_report(y_test, y_pred_xgb, target_names=['ham', 'spam']))

=== XGBoost Classification Report ===
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.94      0.89      0.92       149

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115

